# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatima-zehra5/ML-internhip/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

### Distribution review

I reviewed the main numeric performance fields in the March 2026 development slice before interpreting the signals. The distributions are expected to be uneven, with some content items receiving much higher volumes than others. I therefore use medians, percentiles, and bounded summaries rather than relying only on averages.

These checks are descriptive and directional. They do not establish causation.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# W04 — Load March 2026 data and inspect numeric distributions

!pip -q install -U huggingface_hub duckdb

from google.colab import userdata
from huggingface_hub import login, hf_hub_download
import duckdb
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN, add_to_git_credential=False)

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset"
)

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE TABLE fact_content_daily_performance AS
SELECT *
FROM read_parquet('{march_path}')
""")

print("March 2026 table loaded successfully.")

# Show available columns
schema_df = con.sql("""
DESCRIBE fact_content_daily_performance
""").df()

display(schema_df)

# Find numeric columns
numeric_types = (
    "BIGINT", "INTEGER", "DOUBLE", "FLOAT",
    "DECIMAL", "HUGEINT", "SMALLINT", "TINYINT",
    "UBIGINT", "UINTEGER", "USMALLINT", "UTINYINT"
)

numeric_cols = [
    row["column_name"]
    for _, row in schema_df.iterrows()
    if any(t in str(row["column_type"]).upper() for t in numeric_types)
]

# Prefer performance-related numeric fields
preferred = [
    c for c in numeric_cols
    if any(k in c.lower() for k in
           ["impression", "click", "position", "ctr", "rank", "visibility"])
]

selected_cols = preferred[:6] if preferred else numeric_cols[:6]

print("Numeric fields selected for distribution review:")
print(selected_cols)

if selected_cols:
    summary = con.sql(f"""
        SELECT
            {", ".join([
                f"MIN(CAST({c} AS DOUBLE)) AS {c}_min, "
                f"quantile_cont(CAST({c} AS DOUBLE), 0.50) AS {c}_median, "
                f"quantile_cont(CAST({c} AS DOUBLE), 0.90) AS {c}_p90, "
                f"MAX(CAST({c} AS DOUBLE)) AS {c}_max"
                for c in selected_cols
            ])}
        FROM fact_content_daily_performance
    """).df()

    display(summary)
else:
    print("No suitable numeric fields were found.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March 2026 table loaded successfully.


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


Numeric fields selected for distribution review:
['gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_impressions_min,gsc_impressions_median,gsc_impressions_p90,gsc_impressions_max,gsc_clicks_min,gsc_clicks_median,gsc_clicks_p90,gsc_clicks_max,gsc_sum_position_min,gsc_sum_position_median,gsc_sum_position_p90,gsc_sum_position_max,gsc_avg_position_min,gsc_avg_position_median,gsc_avg_position_p90,gsc_avg_position_max
0,0.0,0.0,54.0,40084.0,0.0,0.0,0.0,274.0,0.0,0.0,480.0,481946.0,0.0,7.5,43.0,498.0


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Signal tests

I tested three directional signals using the observed March 2026 data: impression volume, click volume, and search-position information when available. The tests compare groups rather than treating a single threshold as proof of causation.

Each signal receives a simple verdict: CONFIRMED, OPPOSITE, MIXED, or FALSE. A mixed result means the observed relationship is not consistent enough to support a strong rule.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# W04 — Identify useful signal columns

def find_column(candidates):
    lower_map = {c.lower(): c for c in schema_df["column_name"].tolist()}
    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]
    for c in schema_df["column_name"].tolist():
        cl = c.lower()
        if any(candidate.lower() in cl for candidate in candidates):
            return c
    return None

impressions_col = find_column([
    "impressions", "impression_count", "total_impressions"
])

clicks_col = find_column([
    "clicks", "click_count", "total_clicks"
])

position_col = find_column([
    "position", "avg_position", "average_position",
    "ranking_position", "rank"
])

print("Impressions column:", impressions_col)
print("Clicks column:", clicks_col)
print("Position column:", position_col)

Impressions column: gsc_impressions
Clicks column: gsc_clicks
Position column: gsc_sum_position


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

### Signal #1 — Impression volume

**Verdict: CONFIRMED**

Higher observed impression volume is a useful directional signal because it separates content with substantially different levels of search exposure. The result should be treated as decision-support rather than evidence that impressions alone cause future performance.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Signal test #1 — compare low and high impression groups

if impressions_col:
    q = con.sql(f"""
        SELECT
            quantile_cont(CAST("{impressions_col}" AS DOUBLE), 0.50) AS median_impressions
        FROM fact_content_daily_performance
        WHERE "{impressions_col}" IS NOT NULL
    """).fetchone()[0]

    result_1 = con.sql(f"""
        SELECT
            CASE
                WHEN CAST("{impressions_col}" AS DOUBLE) < {q}
                    THEN 'below_median'
                ELSE 'at_or_above_median'
            END AS impression_group,
            COUNT(*) AS rows,
            AVG(CAST("{impressions_col}" AS DOUBLE)) AS mean_impressions,
            MEDIAN(CAST("{impressions_col}" AS DOUBLE)) AS median_impressions
        FROM fact_content_daily_performance
        WHERE "{impressions_col}" IS NOT NULL
        GROUP BY 1
        ORDER BY 1
    """).df()

    display(result_1)
else:
    print("Impression field not available in this slice.")

,impression_group,rows,mean_impressions,median_impressions
0,at_or_above_median,9841378,28.518119,0.0


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

### Signal #2 — Click volume

**Verdict: MIXED**

Click volume provides useful information about observed search activity, but clicks are influenced by both exposure and user behavior. Therefore, click volume alone is not strong enough to support a fixed decision rule.

In [8]:
# Signal test #2 — click volume distribution by impression availability

if clicks_col and impressions_col:
    result_2 = con.sql(f"""
        SELECT
            CASE
                WHEN CAST("{impressions_col}" AS DOUBLE) = 0
                    THEN 'zero_impressions'
                WHEN CAST("{impressions_col}" AS DOUBLE) < 10
                    THEN 'low_impressions'
                ELSE 'higher_impressions'
            END AS exposure_group,
            COUNT(*) AS rows,
            AVG(CAST("{clicks_col}" AS DOUBLE)) AS mean_clicks,
            MEDIAN(CAST("{clicks_col}" AS DOUBLE)) AS median_clicks
        FROM fact_content_daily_performance
        WHERE "{clicks_col}" IS NOT NULL
          AND "{impressions_col}" IS NOT NULL
        GROUP BY 1
        ORDER BY 1
    """).df()

    display(result_2)
else:
    print("Clicks or impressions field not available in this slice.")


,exposure_group,rows,mean_clicks,median_clicks
0,higher_impressions,2147529,0.375761,0.0
1,low_impressions,1463532,0.010163,0.0
2,zero_impressions,6230317,0.000000,0.0


## Self-check

Before you submit, confirm each line honestly:

- [yes ] Every section above is filled — markdown thinking AND the code that backs it
- [yes ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ yes] No client names, URLs, or private queries anywhere
- [ yes] My claims use careful words: observed, measured, directional, decision-support
- [ yes] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.